In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl
import numpy as np
import sklearn.manifold, sklearn.cluster
import rdkit, rdkit.Chem, rdkit.Chem.Draw
from rdkit.Chem.Draw import IPythonConsole
np.random.seed(0)
import warnings
IPythonConsole.ipython_useSVG = True
warnings.filterwarnings('ignore')
sns.set_context('notebook')
sns.set_style('white',  {'xtick.bottom':True, 'ytick.left':True, 'xtick.color': '#666666', 'ytick.color': '#666666',
                        'axes.edgecolor': '#666666', 'axes.linewidth':     0.8 })
color_cycle = ['#1bbc28', '#F06060', '#5C4B51', '#F3B562', '#6e5687']
mpl.rcParams['axes.prop_cycle'] = mpl.cycler(color=color_cycle) 

In [ ]:
soldata = pd.read_csv('../data/curated-solubility-dataset.csv')
soldata.head()
print (len(soldata))

In [ ]:
soldata

In [ ]:


from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_samples, silhouette_score
from sklearn.cluster import KMeans
from sklearn import datasets, decomposition
from sklearn.manifold import TSNE

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, MACCSkeys, Descriptors, Descriptors3D, Draw, rdMolDescriptors, Draw, PandasTools
from rdkit.DataManip.Metric.rdMetricMatrixCalc import GetTanimotoSimMat, GetTanimotoDistMat
from rdkit.Chem.Draw import IPythonConsole


from math import pi

%config Completer.use_jedi = False
PandasTools.RenderImagesInAllDataFrames(images=True)

In [ ]:
mols = [Chem.MolFromSmiles(smi) for smi in soldata.SMILES]

In [ ]:
#bar=progressbar.ProgressBar(max_value=len(soldata))
table=pd.DataFrame()
for i,mol in enumerate(mols):
    #Chem.SanitizeMol(mol)
    soldata.loc[i,'SMILES']=Chem.MolToSmiles(mol)
    soldata.loc[i,'Mol']=mol
    soldata.loc[i,'NumAliphaticCarbocycles']=Descriptors.NumAliphaticCarbocycles(mol)
    soldata.loc[i,'NumAliphaticHeterocycles']=Descriptors.NumAliphaticHeterocycles(mol)
    soldata.loc[i,'NumAliphaticRings']=Descriptors.NumAliphaticRings(mol)
    soldata.loc[i,'NumAromaticCarbocycles']=Descriptors.NumAromaticCarbocycles(mol)
    soldata.loc[i,'NumAromaticHeterocycles']=Descriptors.NumAromaticHeterocycles(mol)
    soldata.loc[i,'FractionCSP3']=Descriptors.FractionCSP3(mol)
    #bar.update(i+1)

In [ ]:
first_column = soldata.pop('Mol')
soldata.insert(0, 'Mol', first_column)
soldata.head(5)

In [ ]:
soldata

In [ ]:
soldata.drop(soldata[soldata['MolWt'] > 700].index, inplace = True)

In [ ]:
features_start_at = list(soldata.columns).index('MolWt')
feature_names = soldata.columns[features_start_at:]

fig, axs = plt.subplots(nrows=7, ncols=4, sharey=True, figsize=(12, 8), dpi=400)
axs = axs.flatten() # don't want to think about i/j
for i,n in enumerate(feature_names):
    ax = axs[i]
    ax.scatter(
        soldata[n], soldata.Solubility, 
        s = 6, alpha=0.4,
        color = f'C{i}') # add some color 
    if i % 4 == 0:
        ax.set_ylabel('Solubility')
    ax.set_xlabel(n)
# hide empty subplots
for i in range(len(feature_names), len(axs)):
    fig.delaxes(axs[i])
plt.tight_layout()
plt.show()

# Analysis and Visualiazation

In [ ]:
def FPMorganNP(mol):
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2)
    arr = np.zeros((1,))
    DataStructs.ConvertToNumpyArray(fp,arr)
    return arr

In [ ]:
fps = []
for mol in soldata['Mol']:
    fps.append(FPMorganNP(mol))
fps = np.array(fps)

In [ ]:
len(fps)

In [ ]:
from sklearn.manifold import TSNE
tsne = TSNE()
x = tsne.fit_transform(fps)

plt.figure()
plt.scatter(x[:,0],x[:,1],s=2)

In [ ]:
from rdkit.ML.Cluster import Butina

fps = [AllChem.GetMorganFingerprintAsBitVect(mol,2) for mol in soldata['Mol']]

D = []
for i in range(1,len(fps)):
    sims = DataStructs.BulkTanimotoSimilarity(fps[i],list(fps[:i]))
    D.extend([1-a for a in sims])

In [ ]:
plt.figure(figsize=(10,4))

ax1 = plt.subplot(121)
ax1.hist(D, bins = 100, range=(0,1.0));
ax2 = plt.subplot(122)
ax2.hist(D, bins = 100, range=(0,1.0));
ax2.set_yscale('log')

In [ ]:
import numpy as np

bins = [np.log10(x*1e-6) for x in [30,200]]
bins = [-100] + bins + [100]
soldata['bin'] = pd.cut(soldata.Solubility,bins=bins,labels=["Low","Medium","High"])

In [ ]:
color_map_3 = {"Low":"red","Medium":"yellow","High":"green"}
g = sns.displot(x="Solubility",kind="hist",kde=True, height=8, hue="bin",data=soldata,palette=color_map_3)
g.fig.legends[0].set_title("Solubility Bin")

In [ ]:
len(fps)

In [ ]:
soldata['bin'].value_counts()

In [ ]:
desc_columns = soldata.select_dtypes([int,float]).columns[12:]
scaler = StandardScaler()
scaled_descriptors = scaler.fit_transform(soldata[desc_columns])

In [ ]:
tsne3 = TSNE()
tsne_crds3 = tsne.fit_transform(scaled_descriptors)

In [ ]:
ax = sns.scatterplot(x=tsne_crds3[:,0],y=tsne_crds3[:,1],hue=soldata.bin,palette=color_map_3)
ax.get_legend().set_title("Solubility Bin")

In [ ]:
cols = ['Solubility','MolWt', 'MolLogP', 'HeavyAtomCount', 'NumHAcceptors', 
        'NumHDonors','NumRotatableBonds', 'TPSA', 'NumAromaticRings',
        'NumAliphaticCarbocycles','NumAliphaticHeterocycles',
       'NumAromaticCarbocycles','NumAromaticHeterocycles','FractionCSP3'] 

from sklearn.preprocessing import StandardScaler 
stdsc = StandardScaler() 
X_std = stdsc.fit_transform(soldata[cols].iloc[:,range(0,14)].values)

cov_mat=np.cov(X_std.T)
plt.figure(figsize=(12,12))
sns.set(font_scale=1.5)
hm = sns.heatmap(cov_mat,
                 cbar=True,
                 annot=True,
                 square=True,
                 fmt='.2f',
                 annot_kws={'size': 12},
                 cmap='coolwarm',                 
                 yticklabels=cols,
                 xticklabels=cols)
plt.title('Covariance matrix showing correlation coefficients', size = 16)
plt.tight_layout()
plt.show()

depending on the heatmap i have shortlisted few columns for model building

In [ ]:
y = (soldata['Solubility'])
X = (soldata[['MolWt', 'MolLogP', 'HeavyAtomCount', 'NumHAcceptors', 
        'NumRotatableBonds', 'NumAromaticRings',
        'NumAliphaticCarbocycles',
       'NumAromaticCarbocycles','FractionCSP3']])

print(X.shape)
y.shape

In [ ]:


X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(X, y, test_size=0.3)

In [ ]:
import xgboost as xg
xgb_r = xg.XGBRegressor(objective ='reg:linear',
                  n_estimators = 10, seed = 123)

In [ ]:
xgb_r.fit(X_train, y_train)

In [ ]:
xgb_r.score(X_train,y_train)

In [ ]:
xgb_r.score(X_test,y_test)

In [ ]:
pred = xgb_r.predict(X_test)

In [ ]:
from sklearn.metrics import mean_squared_error as MSE

rmse = np.sqrt(MSE(y_test, pred))
print("RMSE : % f" %(rmse))

In [ ]:
MSE(y_test, pred)

In [ ]:
V = (soldata[['MolWt', 'MolLogP', 'HeavyAtomCount', 'NumHAcceptors', 
        'NumRotatableBonds', 'NumAromaticRings',
        'NumAliphaticCarbocycles',
       'NumAromaticCarbocycles','FractionCSP3']])


In [ ]:
X_test

# predictions

In [ ]:
smiless = ['c5ccc(CC4CCN(Cc3nnc(c2cc1ccccc1[nH]2)[nH]3)CC4)cc5','COc4cc3cc(C(=O)NCCN2CCC(Cc1ccccc1)CC2)[nH]c3cc4OC','O=C(NCCN2CCC(Cc1ccccc1)CC2)c4cc3cc(O)ccc3[nH]4']

In [ ]:
def generate(smiles, verbose=False):

    moldata= []
    for elem in smiles:
        mol=Chem.MolFromSmiles(elem) 
        moldata.append(mol)
       
    baseData= np.arange(1,1)
    i=0  
    for mol in moldata:        
       
        desc_MolWt = Descriptors.MolWt(mol)
        desc_MolLogP = Descriptors.MolLogP(mol)
        desc_HeavyAtomCount = rdkit.Chem.Lipinski.HeavyAtomCount(mol)
        desc_NumHAcceptors = rdkit.Chem.Lipinski.NumHAcceptors(mol)
        desc_NumRotatableBonds = Descriptors.NumRotatableBonds(mol)
        desc_NumAromaticRings = rdkit.Chem.Lipinski.NumAromaticRings(mol)
        desc_NumAliphaticCarbocycles = rdkit.Chem.Lipinski.NumAliphaticCarbocycles(mol)
        desc_NumAromaticCarbocycles = rdkit.Chem.Lipinski.NumAromaticCarbocycles(mol)
        desc_FractionCSP3 = rdkit.Chem.Lipinski.FractionCSP3(mol)
           
        row = np.array([desc_MolWt,desc_MolLogP,desc_HeavyAtomCount,desc_NumHAcceptors,
                        desc_NumRotatableBonds,desc_NumAromaticRings,desc_NumAliphaticCarbocycles,
                       desc_NumAromaticCarbocycles,desc_FractionCSP3])   
    
        if(i==0):
            baseData=row
        else:
            baseData=np.vstack([baseData, row])
        i=i+1      
    
    columnNames=["MolLogP","MolWt","NumRotatableBonds",'NumHAcceptors','NumRotatableBonds','NumAromaticRings','NumAliphaticCarbocycles',
                'NumAromaticCarbocycles','FractionCSP3']   
    descriptors = pd.DataFrame(data=baseData,columns=columnNames)
    
    return descriptors

In [ ]:
trial = generate(smiless)

In [ ]:
trial

Prediction of those 3 molecules

In [ ]:
xgb_r.predict(trial)